# Verify the DRAGA monograph
This notebook audits the 289-page book from its own source tree. It reproduces the corpus checks, finite witnesses, C syntax checks, Lean kernel checks and the page comparison. The last section can additionally replay the historical mathematical diagnostics. A CPU runtime is sufficient. The paper experiment has a separate notebook in the repository.

Run inside a Linux checkout or upload a GitHub source ZIP when prompted in Colab. The archive stays in this runtime. On a fresh Colab runtime the notebook installs missing build tools and downloads the pinned Lean distribution. The full historical replay can take considerably longer than the book build.

In [ ]:
import os, sys, shutil, subprocess, tempfile, hashlib, zipfile
from pathlib import Path
work = Path(tempfile.mkdtemp(prefix="draga-proof-notebook-"))
def locate(base):
    candidates = [base, *base.parents]
    return next((p for p in candidates if (p / "monograph/proofs/inventory.json").is_file()), None)

repo = locate(Path(os.environ.get("DRAGA_ROOT", ".")).resolve())
if repo is None:
    from google.colab import files
    print("Upload the repository source ZIP from GitHub.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one repository ZIP.")
    import io
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        for name in archive.namelist():
            path = Path(name)
            if path.is_absolute() or ".." in path.parts:
                raise ValueError("Invalid path in ZIP")
        archive.extractall(work / "checkout")
    inventory, = (work / "checkout").glob("**/monograph/proofs/inventory.json")
    repo = inventory.parents[2]
proof = repo / "monograph/proofs"
def run(args, cwd=proof, **kwargs):
    return subprocess.run([str(x) for x in args], cwd=cwd, check=True, **kwargs)
print(repo)

## Build dependencies
The Python version and the scientific-package versions used for the published verification are listed in the proof guide. The original proof project uses Lean's standard library. The real-valued Fibonacci project adds a pinned mathlib revision. Its first run downloads the locked Git sources and compiled dependencies for the imported modules. Later runs reuse this cache. The Linux x86-64 binary is pinned by release and SHA-256. An existing compatible installation is used directly.

In [ ]:
import importlib.util, platform, urllib.request
needed = {"pymupdf":"pymupdf==1.26.6", "sympy":"sympy==1.14.0",
          "numpy":"numpy", "scipy":"scipy", "mpmath":"mpmath"}
missing = [wheel for module,wheel in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, "-m", "pip", "install", *missing])
if any(shutil.which(tool) is None for tool in ("latexmk", "pdflatex", "cc", "git", "curl")):
    run(["apt-get","-qq","update"])
    run(["apt-get","-qq","install","-y","latexmk","texlive-latex-extra","texlive-science",
         "texlive-fonts-recommended","build-essential","git","curl"])
if not shutil.which("lake") or not shutil.which("lean") or "version 4.19.0" not in subprocess.run(
        ["lean","--version"],cwd=proof/"lean",capture_output=True,text=True).stdout:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Install the toolchain named in lean/lean-toolchain and run again.")
    url = "https://github.com/leanprover/lean4/releases/download/v4.19.0/lean-4.19.0-linux.tar.zst"
    dist = work / "lean.tar.zst"
    urllib.request.urlretrieve(url, dist)
    with dist.open("rb") as file:
        digest = hashlib.file_digest(file, "sha256").hexdigest()
    if digest != "6fe3ce97a58f44e2b3567d455b994eacec5bfe9ae7774f2a573444480ba813fe":
        raise ValueError("Lean release digest mismatch")
    if not shutil.which("zstd"): run(["apt-get","install","-y","zstd"])
    run(["tar","--zstd","-xf",dist,"-C",work])
    os.environ["PATH"] = str(work/"lean-4.19.0-linux/bin") + os.pathsep + os.environ["PATH"]

## Exact proofs and executable witnesses
Lean elaborates the quantified statements and prints each theorem's axiom dependencies. The Fibonacci kernel proof covers every finite history. An independent Python program drives the Lean ordinal codec and compares its state and exact rational midpoint on exhaustive small transitions and selected long traces. The other Python examples enumerate rational or integer instances. The C check restores the original relative header and table paths for each source file.

In [ ]:
import json
run([sys.executable,"checks/formal.py"])
run([sys.executable,"checks/formal.py","--real"])
run([sys.executable,"checks/fibonacci_codec.py"])
run([sys.executable,"checks/reachable.py"],env=dict(os.environ,OPENBLAS_NUM_THREADS="1"))
run([sys.executable,"checks/exact_examples.py"])
run([sys.executable,"applications/verify.py","--integrity-only"])
native = run([sys.executable,"checks/native.py"],capture_output=True,text=True)
print(f"{len(json.loads(native.stdout))} C translation units accepted")

## Recompile and compare every page
The book preserves each original proof byte-for-byte before TeX expansion. The audit resolves labels and bibliography keys, checks the recovered program archive, compares the outline and extracted text, and rasterizes both books at 72 dpi. Identical glyph layout depends on the TeX distribution and installed fonts. A mismatch raises an error and identifies the first affected page.

In [ ]:
build = work/"tex"
build.mkdir()
with (work/"build.log").open("w") as log:
    run(["latexmk","-pdf","-halt-on-error","-interaction=nonstopmode",f"-outdir={build}","monograph.tex"],
        cwd=repo/"monograph",stdout=log,stderr=subprocess.STDOUT)
run([sys.executable,"checks/verify.py","--aux",build/"monograph.aux","--pdf",build/"monograph.pdf"])

## Historical replay
Each entry point is restored to a private temporary directory with its supporting inputs. Results distinguish clean exit, a failed assertion and a time limit. These programs include symbolic identities, finite exhaustive checks and numerical diagnostics. The checked-in verification ledger records the publication run. Set the switch below to rerun all entries or supply a filename substring for a focused run.

In [ ]:
REPLAY_HISTORICAL = False
MATCH = ""  # For example, "verify_complex_hessian.py".
if REPLAY_HISTORICAL:
    run([sys.executable,"checks/replay.py","--match",MATCH,"--jobs","2","--timeout","3600",
         "--output",work/"historical-results.json"])
print(f"Verification outputs: {work}")